# EDA on Student Performance Dataset (Phase 1)

**Name:** Amogh Kumar Sharma
**Reg No:** 23BDS0063

Dataset: [student-mat.csv](https://raw.githubusercontent.com/salemprakash/EDA/main/Data/student-mat.csv) — Student Math Performance dataset.

This notebook covers all the Phase 1 tasks:
1. Loading the dataset
2. Basic statistical analysis
3. Handling missing data
4. Data cleaning
5. Data transformation
6. Univariate analysis
7. Bivariate analysis
8. Multivariate analysis


## 0. Importing required libraries
Basic libraries needed for loading data, doing stats, and plotting.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
%matplotlib inline


## 1. Loading the Dataset
Reading the CSV file directly from the GitHub raw link given in the assignment.

In [ ]:
url = "https://raw.githubusercontent.com/salemprakash/EDA/main/Data/student-mat.csv"

# the file is actually semicolon separated (;) and not comma separated,
# so we need to mention sep=';' otherwise all columns will load as one column
df = pd.read_csv(url, sep=';')

# checking first 5 rows to see if it loaded properly
df.head()


In [ ]:
# checking shape of dataset -> (rows, columns)
print("Shape of dataset:", df.shape)

# checking column names
print("\nColumn names:")
print(df.columns.tolist())


In [ ]:
# quick info about datatypes and non-null counts
df.info()


## 2. Applying Basic Statistical Analysis
Using `describe()` to get mean, std, min, max, quartiles etc. for the numeric columns.
Also checking value counts for a few categorical columns.

In [ ]:
# statistical summary of all numeric columns
df.describe()


In [ ]:
# describe() for categorical columns too
df.describe(include='object')


In [ ]:
# checking some individual stats for the final grade column (G3)
print("Mean of G3:", df['G3'].mean())
print("Median of G3:", df['G3'].median())
print("Mode of G3:", df['G3'].mode()[0])
print("Std Dev of G3:", df['G3'].std())
print("Variance of G3:", df['G3'].var())


## 3. Handling Missing Data
Checking if there are any missing/null values in the dataset. This particular dataset is usually clean
(no missing values), but we still check properly instead of assuming.

In [ ]:
# checking null values column-wise
print("Missing values per column:")
print(df.isnull().sum())

print("\nTotal missing values in dataset:", df.isnull().sum().sum())


In [ ]:
# Even though this dataset does not have missing values,
# writing the handling code anyway so the pipeline is complete
# for numeric columns -> fill missing with median.
num_cols = df.select_dtypes(include=np.number).columns
for col in num_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())

# for categorical columns -> fill missing with mode
cat_cols = df.select_dtypes(include='object').columns
for col in cat_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

# confirming no missing values remain
print("Missing values after handling:", df.isnull().sum().sum())


## 4. Data Cleaning
Checking for duplicate rows and any obvious inconsistencies in the data.

In [ ]:
# checking for duplicate rows
print("Number of duplicate rows:", df.duplicated().sum())

# dropping duplicates if any
df = df.drop_duplicates()
print("Shape after removing duplicates:", df.shape)


In [ ]:
# checking unique values in some categorical columns to spot any weird/inconsistent entries
for col in ['school', 'sex', 'address', 'famsize', 'Pstatus', 'higher', 'internet']:
    print(col, "->", df[col].unique())


In [ ]:
# stripping any extra spaces from column names,
df.columns = df.columns.str.strip()

# resetting index after dropping duplicates so index stays continuous
df = df.reset_index(drop=True)


## 5. Data Transformation
Creating a couple of new/derived columns and encoding categorical variables so that they
can be used later for analysis (like correlation, multivariate plots etc.)

In [ ]:
# Grades G1, G2, G3 are on a 0-20 scale, adding an average grade column
df['avg_grade'] = (df['G1'] + df['G2'] + df['G3']) / 3
# just a simple bucketing: Low / Medium / High
def grade_bucket(g):
    if g < 10:
        return 'Low'
    elif g < 15:
        return 'Medium'
    else:
        return 'High'

df['performance'] = df['G3'].apply(grade_bucket)

df[['G1', 'G2', 'G3', 'avg_grade', 'performance']].head()


In [ ]:
# label encoding some binary categorical columns (yes/no -> 1/0) so they're easier to use in plots/corr
binary_cols = ['schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic']

for col in binary_cols:
    df[col + '_enc'] = df[col].map({'yes': 1, 'no': 0})

df[binary_cols + [c + '_enc' for c in binary_cols]].head()


## 6. Univariate Analysis
Looking at individual columns one at a time. Using histograms, count plots and a box plot.

In [ ]:
# Plot 1: Histogram of final grade (G3)
plt.figure(figsize=(7,5))
sns.histplot(df['G3'], bins=15, kde=True, color='skyblue')
plt.title("Distribution of Final Grade (G3)")
plt.xlabel("Final Grade (G3)")
plt.ylabel("Number of Students")
plt.show()


In [ ]:
# Plot 2: Count plot of students by sex
plt.figure(figsize=(6,4))
sns.countplot(x='sex', data=df, palette='pastel')
plt.title("Count of Students by Gender")
plt.xlabel("Sex")
plt.ylabel("Count")
plt.show()


In [ ]:
# Plot 3: Box plot for age to check spread and outliers
plt.figure(figsize=(6,4))
sns.boxplot(x=df['age'], color='lightgreen')
plt.title("Boxplot of Student Age")
plt.xlabel("Age")
plt.show()


In [ ]:
# Plot 4 (extra): Distribution of weekly study time
plt.figure(figsize=(6,4))
sns.countplot(x='studytime', data=df, palette='muted')
plt.title("Weekly Study Time Distribution")
plt.xlabel("Study Time (1=low ... 4=high)")
plt.ylabel("Count")
plt.show()


## 7. Bivariate Analysis
Looking at relationship between two variables at a time.

In [ ]:
# Plot 1: Study time vs Final grade
plt.figure(figsize=(7,5))
sns.boxplot(x='studytime', y='G3', data=df, palette='Set2')
plt.title("Study Time vs Final Grade")
plt.xlabel("Study Time (1=low ... 4=high)")
plt.ylabel("Final Grade (G3)")
plt.show()


In [ ]:
# Plot 2: Scatter plot - G1 vs G3 (first period grade vs final grade)
plt.figure(figsize=(7,5))
sns.scatterplot(x='G1', y='G3', data=df, hue='sex')
plt.title("First Period Grade (G1) vs Final Grade (G3)")
plt.xlabel("G1")
plt.ylabel("G3")
plt.show()


In [ ]:
# Plot 3: Average final grade by parents' cohabitation status
plt.figure(figsize=(6,4))
sns.barplot(x='Pstatus', y='G3', data=df, palette='coolwarm', ci=None)
plt.title("Parents' Cohabitation Status vs Average Final Grade")
plt.xlabel("Pstatus (T = together, A = apart)")
plt.ylabel("Average Final Grade")
plt.show()


In [ ]:
# Plot 4 (extra): Correlation between absences and final grade
plt.figure(figsize=(7,5))
sns.scatterplot(x='absences', y='G3', data=df, color='purple')
plt.title("Absences vs Final Grade")
plt.xlabel("Number of Absences")
plt.ylabel("Final Grade (G3)")
plt.show()


## 8. Multivariate Analysis
Looking at relationships between 3 or more variables together.

In [ ]:
# Plot 1: Correlation heatmap of numeric columns
plt.figure(figsize=(12,9))
corr = df.select_dtypes(include=np.number).corr()
sns.heatmap(corr, cmap='coolwarm', annot=False)
plt.title("Correlation Heatmap of Numeric Features")
plt.show()


In [ ]:
# Plot 2: Study time vs G3, split by gender (hue) and internet access (col)
g = sns.catplot(x='studytime', y='G3', hue='sex', col='internet',
                 data=df, kind='box', palette='Set2', height=5, aspect=0.9)
g.fig.suptitle("Study Time vs G3 by Gender and Internet Access", y=1.05)
plt.show()


In [ ]:
# Plot 3: Pairplot of a few important numeric columns, colored by performance bucket
cols_to_plot = ['G1', 'G2', 'G3', 'studytime', 'absences', 'performance']
sns.pairplot(df[cols_to_plot], hue='performance', palette='husl')
plt.suptitle("Pairplot of Key Variables by Performance Level", y=1.02)
plt.show()


In [ ]:
# Plot 4 : Avg final grade by study time and family support, grouped bar chart
pivot_df = df.groupby(['studytime', 'famsup'])['G3'].mean().reset_index()

plt.figure(figsize=(8,5))
sns.barplot(x='studytime', y='G3', hue='famsup', data=pivot_df, palette='Set1')
plt.title("Avg Final Grade by Study Time and Family Support")
plt.xlabel("Study Time (1=low ... 4=high)")
plt.ylabel("Average Final Grade")
plt.show()


## Conclusion (Phase 1)

- The dataset had no major missing values, but the missing-value handling step was still included for completeness.
- Duplicate rows were checked and removed (if any existed).
- New columns (`avg_grade`, `performance`) were created during transformation to help with later analysis.
- From the univariate plots, most students score in the mid-range for final grades, and study time is fairly skewed towards lower values (1-2).
- From the bivariate plots, there seems to be a positive relation between study time and final grade, and a strong positive relation between G1/G2 and G3 (expected, since they are earlier grades in the same subject).
- The correlation heatmap and pairplot in the multivariate section show G1, G2 and G3 are highly correlated with each other, while other features like absences have weaker correlation with final grade.

This completes all Phase 1 requirements.
